# Case Assignment #2
## Sarthak Jain, MSIS 510 B

In [122]:
# Add imports
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

In [123]:
# Load the dataset
df = pd.read_csv("Parole.csv")

### a) How many parolees do we have data for? Of the parolees that we have data for, what percentage violated the terms of their parole?

Number of parolees we have data for: 675   
Of the parolees we have data for, the percentage violated the terms of their parole: 11.56%

In [124]:
# Parolees we have data for
parolee_count = df.shape[0]
print(f"Number of parolees in the dataset: {parolee_count}")

Number of parolees in the dataset: 675


In [125]:
# Percentage of parolees who violated parole
violation_percentage = (df['Violator'].sum() / parolee_count) * 100
print(f"Percentage of parolees who violated parole: {violation_percentage:.2f}%")

Percentage of parolees who violated parole: 11.56%


### b) Randomly split the data into a training set and a test set, putting 70% of the data in the training set. Then, build a logistic regression model to predict the variable `Violator` using all of the other variables as independent variables. You should use the training dataset to build the model. 

In [126]:
# Split the dataset into training and testing sets
df_train, df_test = train_test_split(df, test_size=0.3, random_state=42)

In [127]:
# Train a logistic regression model
model = smf.logit(formula='Violator ~ Male + RaceWhite + Age + State + TimeServed + MaxSentence + MultipleOffenses + Crime', data=df_train).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.244205
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:               Violator   No. Observations:                  472
Model:                          Logit   Df Residuals:                      459
Method:                           MLE   Df Model:                           12
Date:                Sun, 15 Nov 2026   Pseudo R-squ.:                  0.2573
Time:                        21:08:30   Log-Likelihood:                -115.26
converged:                       True   LL-Null:                       -155.19
Covariance Type:            nonrobust   LLR p-value:                 4.409e-12
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.7217      1.403     -0.515      0.607      -3.471       2.027
State

#### i) Describe your resulting model. What variables are significant in your model?

In [128]:
# The R^2 for the training data. Need SSE and SST in order to calculate R^2
# a) R-squared value on training data
train_predictions = model.predict(df_train)
train_actual = df_train['Violator']

# Calculate SSE and SST for training data
train_mean = train_actual.mean()
train_sst = ((train_actual - train_mean) ** 2).sum()
train_sse = ((train_actual - train_predictions) ** 2).sum()

# Calculate R-squared for training data
train_r2 = 1 - (train_sse / train_sst)
print(f"Training Data R-squared: {train_r2:.4f}")


# The R^2 for the testing data. Need SSE and SST in order to calculate R^2
test_predictions = model.predict(df_test)
test_actual = df_test['Violator']

# Calculate SST and SSE for test data
test_sst = ((test_actual - train_mean) ** 2).sum()
test_sse = ((test_actual - test_predictions) ** 2).sum()

# Calculate R-squared for test data
test_r2 = 1 - (test_sse / test_sst)
print(f"Test Data R-squared: {test_r2:.4f}")


#Statistically significant features
p_values = model.pvalues

for feature, p_value in p_values.items():
    if p_value < 0.05:
        print(f"The feature '{feature}' is statistically significant (p-value: {p_value:.4f})")


Training Data R-squared: 0.2190
Test Data R-squared: 0.2290
The feature 'State[T.Virginia]' is statistically significant (p-value: 0.0000)
The feature 'MultipleOffenses' is statistically significant (p-value: 0.0006)


The model is statistically significant for the feature State[T.Virginia] and for MultipleOffenses. The training data R<sup>2</sup> is 0.2190. The testing data R<sup>2</sup> is 0.2290

#### ii) Consider a parolee who is male, of white race, aged 50 years, at a prison release, from the state of Maryland, served three months, had a maximum sentence of 12 months, did not commit multiple offenses, and committed a larceny. According to your model, what is the probability that this individual is a violator?

In [129]:
parolee_scenario = {
    'Male': 1,
    'RaceWhite': 1,
    'Age': 50.0,
    'State': 'Other',
    'TimeServed': 3.0,
    'MaxSentence': 12,
    'MultipleOffenses': 0,
    'Crime': 'Larceny'
}
parolee_violator_prob = model.predict(parolee_scenario)

print(f"Predicted profitability is: {parolee_violator_prob[0]:,.5f}")

Predicted profitability is: 0.10895


The predicted probability that a white male from Maryland who served three months out of his twelve month sentence for larceny and no other multiple crime being a violator is 0.10895 or 10.895%

#### iii) Now compute the model's predicted probabilities for parolees in the testing set. Then create a confusion matrix for the test set using a threshold of 0.5. What is the model's false positive rate on the test set? False negative rate? Overall accuracy?

In [130]:
y_pred_prob_test = model.predict(df_test)
y_pred_test = (y_pred_prob_test> 0.5)*1

cm = confusion_matrix(df_test['Violator'], y_pred_test)
print(cm)
true_negative, false_positive, false_negative, true_positive = cm.ravel()

false_positive_rate = false_positive / (false_positive + true_negative)
print(f"False Positive Rate is: {false_positive_rate:.4f}")

false_negative_rate = false_negative / (false_negative + true_positive)
print(f"False Negative Rate is: {false_negative_rate:.4f}")

overall_accuracy = (true_negative + true_positive) / (len(df_test))
print(f"Overall accuracy is {overall_accuracy:.4f}")


[[169   4]
 [ 23   7]]
False Positive Rate is: 0.0231
False Negative Rate is: 0.7667
Overall accuracy is 0.8670


The false positive rate is 0.0231 or 2.31%
The false negative rate is 0.7667 or 76.67%
The overall accuracy is 0.8670 or 86.70%

#### iv) Compare your accuracy on the test set to a _baseline_ model that predicts every parolee in the test set is a non-violator, regardless of the values of the independent variables. Does your model improve over this simple model?

In [131]:
baseline_model = (df_test['Violator'] == 0).mean()
print(f"The baseline model predicting every non-violator in the test set is: {baseline_model:.4f}")

print(f"The model accuracy is: {overall_accuracy:.4f}")

The baseline model predicting every non-violator in the test set is: 0.8522
The model accuracy is: 0.8670


The baseline model is 0.8522 or 85.22%, whereas the model accuracy is 0.8670 or 86.70%. My model slightly improves over the simple model.

#### v) Consider a parole board that might use your model to predict where parolees will be violators or not. The job of a parole board is to make sure that a prisoner is ready to be released into free society, and therefore parole boards tend to be particularly concerned about releasing prisoners who will violate their parole. Would the parole board be more concerned by false positive errors or false negative errors? How should they adjust their threshold to reflect their error preferences? 

Answer - The parole board should be concerned with false negative errors since that's when the board would release prisoners who would violate their parole. To adjust the threshold to reflect the error preferences, the board should lower the threshold from 0.5 to something smaller, but there is then the trade off of losing overall model accuracy and increasing the false positive rate. See an example below of what it would like if the threshold was lowered to 0.25, where the false negative rate decreases, but the false positive rate increases lowering the accuracy overall too.

In [132]:
y_pred_prob_test_025 = model.predict(df_test)
y_pred_test_025 = (y_pred_prob_test_025> 0.25)*1

cm_025 = confusion_matrix(df_test['Violator'], y_pred_test_025)
print(cm_025)
true_negative_025, false_positive_025, false_negative_025, true_positive_025 = cm_025.ravel()

false_positive_rate_025 = false_positive_025 / (false_positive_025 + true_negative_025)
print(f"False Positive Rate is: {false_positive_rate_025:.4f}")

false_negative_rate_025 = false_negative_025 / (false_negative_025 + true_positive_025)
print(f"False Negative Rate is: {false_negative_rate_025:.4f}")

overall_accuracy_025 = (true_negative_025 + true_positive_025) / (len(df_test))
print(f"Overall accuracy is {overall_accuracy_025:.4f}")


[[154  19]
 [ 18  12]]
False Positive Rate is: 0.1098
False Negative Rate is: 0.6000
Overall accuracy is 0.8177


#### vi) Compute the AUC of the model on the test set, and interpret what the number means in this context. Considering the AUC, the accuracy compared to the baseline model, and what happens when the threshold is adjusted, do you think this model is of value to a parole board? Why or why not?

In [133]:
# AUC on the test set
auc_baseline = roc_auc_score(df_test['Violator'], y_pred_prob_test)
print(f"Test AUC is {auc_baseline}")

auc_025 = roc_auc_score(df_test['Violator'], y_pred_prob_test_025)
print(f"Test AUC with threshold 0.25 is {auc_025}")



Test AUC is 0.8564547206165704
Test AUC with threshold 0.25 is 0.8564547206165704


The AUC of the test set is 0.856, and it means that there is 85.6% chance the model is accurate in identifying a higher-risk parolee over a lower-risk parolee. This is helpful to the parole board, but not by much. The accuracy is 0.8670 of the model and the AUC is 0.856, so the difference is minimal, and there isn't much more on here. 

If the threshold is adjusted, the AUC stays the same, which seems odd, as the AUC should have changed as the accuracy of the model did as the threshold changed. Based on that alone, the board should use this to supplement their judgment but not go solely off of this.

### c) Our goal in this problem has been to predict the outcome of a parole decision, and we used a publicly available dataset of parole releases for predictions. It is always important to evaluate a dataset for possible sources of bias, especially when the dataset only contains a subset of the observations of interest. 

### The _Parole.csv_ dataset contains all individuals released from parole in 2004, either due to completing their parole term or violating the terms of their parole. However, it doesn't contain parolees who neither violated their parole nor completed their term in 2004, causing non-violators to be underrepresented. This is called _selection bias_ or _selecting on the dependent variable_, because we used our dependent variable (parole violation) to select only a subset of all relevant parolees to include in our analysis. How could we improve our dataset to best address selection bias?

**Answer**
One way to address the selection bias would be by having everyone who was released on parole in 2004 with a new extra field to track them. Anyone violating their parole would be marked as `Violator`. Anyone who doesn't won't be marked as a `Violator`.  There can be another field for `Complete` where if someone completes their parole term without violating their term, they would be marked with a 1 otherwise, with a 0. We could also find a way to only represent those that have completed their parole period in the dataset, reducing the risk of ambiguity.